# SOLSTICE GNN state models: DIII-D

Train the graph models on the DIII-D APP-FPP store, using the `solstice`
library itself (models, graph builders):

- **`gnn_v1`** — conditional GNN on the native SOLPS mesh (FiLM
  conditioning per message-passing layer; solpex-paper baseline)
- **`gnn_encproc`** — encode-process-decode with a coarse latent mesh
  (WeatherNext/anemoi-style; long-range information travels across the
  latent mesh instead of one cell per layer)

One model predicts **all fields jointly** (multi-output), unlike the
per-field MLPs of the quickstart notebook — compare against those
metrics. Needs the store in Drive and a GitHub token in Colab secrets
(key `colab`) since the repo is private.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('colab')   # Colab secret holding a GitHub token
!pip -q install git+https://{token}@github.com/abdoudiaw/solstice.git torch_geometric xarray netcdf4

STORE = '/content/drive/MyDrive/SOLPS_DATA/solstice_store_diiid_appfpp_v1.nc'
WEIGHTS_DEST = '/content/drive/MyDrive/SOLPS_DATA/solstice_weights/gnn'

# ---- knobs ----
MODEL    = 'gnn_encproc'   # 'gnn_v1' | 'gnn_encproc'
HIDDEN   = 128
LAYERS   = 8               # message-passing / processor layers
N_LATENT = 256             # latent mesh size (gnn_encproc only)
EPOCHS   = 300
BATCH    = 32              # cases per step
LR       = 1e-3


In [ ]:
import numpy as np, xarray as xr, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from solstice.graphs import cell_adjacency_edges, default_node_features, build_latent_graph
from solstice.models import build_model

ds = xr.open_dataset(STORE)
N_CELLS = ds.sizes['cell']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(dict(ds.sizes), '| device:', device)


## Inputs (same feature engineering as the MLP notebook) and targets


In [ ]:
FIELDS = {'te': True, 'ti': True, 'ne': True, 'na_D0': True, 'na_D1': True,
          'ua_D1': False, 'q_pol': False, 'prad': True}  # name -> log10?
OUT = list(FIELDS)

raw = {v[6:]: ds[v].values.astype(np.float64) for v in ds.data_vars if v.startswith('input_')}
for name in list(raw):
    if np.unique(raw[name]).size == 1:
        print(f'dropping constant input: {name}'); del raw[name]
if 'pe' in raw and 'pi' in raw and np.array_equal(raw['pe'], raw['pi']):
    raw['ptot'] = raw.pop('pe') + raw.pop('pi'); print('merged pe + pi -> ptot')
if 'hci' in raw and 'hce' in raw and np.array_equal(raw['hci'], raw['hce']):
    raw['chi'] = raw.pop('hci'); del raw['hce']; print('merged hci = hce -> chi')
INPUTS = sorted(raw)
LOG_INPUTS = ('core_fueling', 'puff_D2', 'puff_Ne')
X = np.stack([raw[v] for v in INPUTS], axis=1)
for j, v in enumerate(INPUTS):
    if v in LOG_INPUTS:
        X[:, j] = np.log10(np.clip(X[:, j], 1e-30, None))
x_mean, x_std = X.mean(0), X.std(0) + 1e-12
Xn = (X - x_mean) / x_std
print('model inputs:', INPUTS)

# targets: (case, cell, field), per-cell-per-field standardization
Y, y_mean, y_std = [], [], []
for name in OUT:
    y = ds[name].values.astype(np.float64)
    if FIELDS[name]:
        y = np.log10(np.clip(np.abs(y), 1e-6, None))
    m, s = y.mean(0), y.std(0) + 1e-12
    Y.append((y - m) / s); y_mean.append(m); y_std.append(s)
Y = np.stack(Y, axis=2)
y_mean, y_std = np.stack(y_mean, 1), np.stack(y_std, 1)

rng = np.random.default_rng(0)
idx = rng.permutation(ds.sizes['case'])
split = int(0.85 * len(idx))
itr, ite = idx[:split], idx[split:]
print(len(itr), 'train /', len(ite), 'test cases')


## Graph construction (`solstice.graphs`)


In [ ]:
x_nodes = torch.tensor(default_node_features(ds), device=device)
x_nodes = (x_nodes - x_nodes.mean(0)) / (x_nodes.std(0) + 1e-12)
ei_np, ea_np = cell_adjacency_edges(ds)
ea_np = ea_np / np.abs(ea_np).max(axis=0)
print('cells:', N_CELLS, 'edges:', ei_np.shape[1])

if MODEL == 'gnn_encproc':
    lg = build_latent_graph(ds.cell_r.values, ds.cell_z.values, n_latent=N_LATENT, k_nn=6)
    la_np = lg['latent_attr'] / np.abs(lg['latent_attr']).max(axis=0)
    aa_np = lg['assign_attr'] / (np.abs(lg['assign_attr']).max(axis=0) + 1e-12)
    print('latent nodes:', N_LATENT, 'latent edges:', lg['latent_edges'].shape[1])


## Batched training (replicated graph, offset indices)


In [ ]:
def batch_graph(case_ids):
    '''Stack B copies of the fixed graph; returns model kwargs + target.'''
    B = len(case_ids)
    xb = x_nodes.repeat(B, 1)
    pb = torch.tensor(Xn[case_ids], dtype=torch.float32, device=device)
    yb = torch.tensor(Y[case_ids].reshape(B * N_CELLS, -1), dtype=torch.float32, device=device)
    if MODEL == 'gnn_v1':
        offs = (np.arange(B)[:, None] * N_CELLS)
        ei = torch.tensor(np.concatenate([ei_np + o for o in offs.ravel()], axis=1), device=device)
        ea = torch.tensor(np.tile(ea_np, (B, 1)), dtype=torch.float32, device=device)
        params = pb.repeat_interleave(N_CELLS, dim=0)
        return dict(x=xb, edge_index=ei, edge_attr=ea, params=params), yb
    ai = torch.tensor(np.concatenate(
        [lg['assign_index'] + np.array([[b * N_CELLS], [b * N_LATENT]]) for b in range(B)], axis=1), device=device)
    aa = torch.tensor(np.tile(aa_np, (B, 1)), dtype=torch.float32, device=device)
    le = torch.tensor(np.concatenate([lg['latent_edges'] + b * N_LATENT for b in range(B)], axis=1), device=device)
    la = torch.tensor(np.tile(la_np, (B, 1)), dtype=torch.float32, device=device)
    pl = pb.repeat_interleave(N_LATENT, dim=0)
    return dict(x=xb, assign_index=ai, assign_attr=aa, latent_edges=le,
                latent_attr=la, params_latent=pl, n_latent=B * N_LATENT), yb

cfg = {'node_features': 2, 'param_dim': len(INPUTS), 'out_features': len(OUT), 'hidden': HIDDEN}
cfg |= {'n_layers': LAYERS} if MODEL == 'gnn_v1' else {'n_process_layers': LAYERS}
model = build_model(MODEL, cfg).to(device)
print(MODEL, sum(p.numel() for p in model.parameters()), 'parameters')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
kw_val, y_val = batch_graph(ite)
best, best_state = np.inf, None
for ep in range(EPOCHS):
    model.train()
    perm = rng.permutation(itr)
    tr_loss = 0.0
    for i in range(0, len(perm), BATCH):
        kw, yb = batch_graph(perm[i:i + BATCH])
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(**kw), yb)
        loss.backward(); opt.step()
        tr_loss += loss.item() * len(perm[i:i + BATCH])
    sched.step()
    model.eval()
    with torch.no_grad():
        val = nn.functional.mse_loss(model(**kw_val), y_val).item()
    if val < best:
        best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
    if ep % 10 == 0:
        print(f'ep{ep}: train {tr_loss/len(perm):.4f} val {val:.4f}')
model.load_state_dict(best_state)
print('best val MSE (normalized):', round(best, 5))


## Metrics (same definitions as the MLP notebook — compare tables)


In [ ]:
from skimage.metrics import structural_similarity as ssim
import json as _json
FACE_SETS = _json.loads(ds.attrs['face_sets'])
NX, NY = ds.attrs['nx'], ds.attrs['ny']

def to_image(v):
    img = np.full((NX, NY), np.nan)
    img[ds.cell_ix.values - 1, ds.cell_iy.values - 1] = v
    return img

def target_cells(which):
    faces = ds.face_set.values == FACE_SETS[which]
    cells = ds.face_cells.values[faces, 0]
    return cells[np.argsort(ds.cell_iy.values[cells])]
INNER, OUTER = target_cells('inner_target'), target_cells('outer_target')

model.eval()
with torch.no_grad():
    kw, _ = batch_graph(ite)
    P_all = model(**kw).cpu().numpy().reshape(len(ite), N_CELLS, len(OUT))
T_all = Y[ite]

print(f"{'field':8s} {'SSIM':>6s} {'R2':>7s} {'RMSE':>9s} {'peak-tgt %':>11s}")
for j, name in enumerate(OUT):
    P = P_all[:, :, j] * y_std[:, j] + y_mean[:, j]
    T = T_all[:, :, j] * y_std[:, j] + y_mean[:, j]
    ss = np.mean([ssim(to_image(t), to_image(p), data_range=np.nanmax(t)-np.nanmin(t))
                  for t, p in zip(T, P)])
    r2 = 1 - np.sum((P - T)**2) / np.sum((T - T.mean())**2)
    rmse = np.sqrt(np.mean((P - T)**2))
    peak = np.median([abs(p[c].max() - t[c].max()) / abs(t[c].max()) * 100
                      for t, p in zip(T, P) for c in (INNER, OUTER)])
    unit = 'dex' if FIELDS[name] else 'phys'
    print(f'{name:8s} {ss:6.3f} {r2:7.3f} {rmse:9.3f} ({unit}) {peak:9.2f}')


## Plots: 2D, targets, OMP


In [ ]:
def phys(j, arr):
    y = arr * y_std[:, j] + y_mean[:, j]
    return 10**y if FIELDS[OUT[j]] else y

def plot_field(values, title='', ax=None, cmap='viridis', sym=False):
    verts = np.stack([ds.cell_corners_r.values, ds.cell_corners_z.values], axis=-1)
    ax = ax or plt.subplots(figsize=(4, 6))[1]
    kwp = dict(clim=(-np.max(np.abs(values)), np.max(np.abs(values)))) if sym else {}
    pc = PolyCollection(verts, array=values, cmap=cmap, edgecolor='none', **kwp)
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_title(title); plt.colorbar(pc, ax=ax, shrink=0.8)

kk = 0  # index into test set
j_te = OUT.index('te')
pred_te = phys(j_te, P_all[kk, :, j_te])
true_te = phys(j_te, T_all[kk, :, j_te])
fig, axs = plt.subplots(1, 3, figsize=(13, 6))
plot_field(true_te, 'Te SOLPS (eV)', axs[0])
plot_field(pred_te, f'Te {MODEL} (eV)', axs[1])
plot_field(pred_te - true_te, 'error (eV)', axs[2], cmap='RdBu_r', sym=True)
plt.tight_layout()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, (cells, label) in zip(axs, [(INNER, 'inner target'), (OUTER, 'outer target'), (None, 'OMP')]):
    if cells is None:
        lfs = ds.cell_r.values > ds.cell_r.values.mean()
        ix_omp = ds.cell_ix.values[lfs][np.argmin(np.abs(ds.cell_z.values[lfs]))]
        cells = np.where(ds.cell_ix.values == ix_omp)[0]
        cells = cells[np.argsort(ds.cell_iy.values[cells])]
    r = ds.cell_r.values[cells]
    ax.plot(r, true_te[cells], 'o-', label='SOLPS')
    ax.plot(r, pred_te[cells], 's--', label=MODEL)
    ax.set_yscale('log'); ax.set_title(f'{label} Te (eV)'); ax.set_xlabel('R (m)'); ax.legend()
plt.tight_layout()


## Cost and save


In [ ]:
import time, os
kw1, _ = batch_graph(ite[:1])
with torch.no_grad():
    for _ in range(10): model(**kw1)
    if device == 'cuda': torch.cuda.synchronize()
    ts = []
    for _ in range(100):
        t0 = time.perf_counter(); model(**kw1)
        if device == 'cuda': torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
lat = np.median(ts) * 1e3
n_par = sum(p.numel() for p in model.parameters())
print(f'{MODEL}: {n_par:,} params, {n_par*4/1e6:.1f} MB, {lat:.3f} ms/case (all {len(OUT)} fields jointly)')
print('vs REACT 1 ms budget ->', 'WITHIN' if lat < 1.0 else 'OVER')

os.makedirs(WEIGHTS_DEST, exist_ok=True)
torch.save({'state_dict': model.state_dict(), 'model_class': MODEL, 'config': cfg,
            'fields': FIELDS, 'inputs': INPUTS, 'x_mean': x_mean, 'x_std': x_std,
            'y_mean': y_mean, 'y_std': y_std,
            'n_latent': N_LATENT if MODEL == 'gnn_encproc' else None,
            'epochs': EPOCHS, 'seed': 0},
           f'{WEIGHTS_DEST}/diiid-appfpp-state-{MODEL}.pt')
print(os.listdir(WEIGHTS_DEST))
